In [ ]:
!pip install -q transformers
!pip install -q pytorch-lightning
!pip install -q wandb
!pip install -q torchmetrics
!pip install -q bitsandbytes

In [ ]:
import warnings
warnings.filterwarnings("ignore")
from tqdm.auto import tqdm
import pandas as pd
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import pytorch_lightning as pl
from transformers import AutoTokenizer, AutoModelForCausalLM, get_cosine_schedule_with_warmup, BitsAndBytesConfig
from torch.optim import AdamW
from torchmetrics.text.rouge import ROUGEScore
from torchmetrics.text import BLEUScore
from pytorch_lightning import Trainer
import wandb
from sklearn.model_selection import train_test_split
import gc
import os

In [ ]:
# Fill in your W&B API key and project name before running
# Get your key at https://wandb.ai/authorize
WANDB_KEY = ''      # e.g. 'abc123...'
WANDB_PROJECT = ''  # e.g. 'gec-simpo-gemma'
MAX_CTX_CHARS = 50  # characters kept from each context sentence (~20 tokens)
wandb.login(key=WANDB_KEY)
pl.seed_everything(17)


In [ ]:
class PreferenceDataset(Dataset):
    def __init__(self, dataframe, tokenizer, max_len=192):
        self.dataframe = dataframe.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_len = max_len
        
    def __len__(self):
        return len(self.dataframe)
    
    def create_prompt(self, input_text, prev_sent='', next_sent=''):
        p   = str(prev_sent)[:MAX_CTX_CHARS].strip()
        n   = str(next_sent)[:MAX_CTX_CHARS].strip()
        ctx = ''
        if p:
            ctx += f'Предыдущее: {p}\n'
        if n:
            ctx += f'Следующее: {n}\n'
        return f'{ctx}Исправь все ошибки в предложении: {input_text}\nОтвет: '
    
    def __getitem__(self, index):
        row = self.dataframe.iloc[index]
        input_text = str(row['input']) or "<empty>"
        correct_output = str(row['output']) or "<empty>"
        prev_sent = str(row.get('prev_sent', '') or '')
        next_sent = str(row.get('next_sent', '') or '')

        prompt = self.create_prompt(input_text, prev_sent, next_sent)
        chosen_text = prompt + correct_output
        rejected_text = prompt + input_text
        chosen_tokens = self.tokenizer(
            chosen_text,
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        
        rejected_tokens = self.tokenizer(
            rejected_text,
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        
        prompt_tokens = self.tokenizer(
            prompt,
            max_length=self.max_len,
            padding=False,
            truncation=True,
            return_tensors='pt'
        )
        
        return {
            'chosen_input_ids': chosen_tokens['input_ids'].squeeze(),
            'chosen_attention_mask': chosen_tokens['attention_mask'].squeeze(),
            'rejected_input_ids': rejected_tokens['input_ids'].squeeze(),
            'rejected_attention_mask': rejected_tokens['attention_mask'].squeeze(),
            'prompt_length': len(prompt_tokens['input_ids'].squeeze()),
            'input_text': input_text,
            'correct_output': correct_output,
            'prompt': prompt
        }

In [ ]:
class SimPOLightningModule(pl.LightningModule):
    def __init__(self, model_name="Vikhrmodels/Vikhr-Gemma-2B-instruct", learning_rate=1e-5,
                 beta=2.0, total_steps=None, gradient_checkpointing=True, use_quantization=True):
        super(SimPOLightningModule, self).__init__()
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        if self.tokenizer.pad_token is None:
            self.tokenizer.pad_token = self.tokenizer.eos_token

        if use_quantization:
            bnb_config = BitsAndBytesConfig(
                load_in_4bit=True,
                bnb_4bit_use_double_quant=True,
                bnb_4bit_quant_type="nf4",
                bnb_4bit_compute_dtype=torch.bfloat16
            )
            self.model = AutoModelForCausalLM.from_pretrained(
                model_name,
                quantization_config=bnb_config,
                device_map="auto",
                trust_remote_code=True,
                torch_dtype=torch.bfloat16
            )
        else:
            self.model = AutoModelForCausalLM.from_pretrained(
                model_name,
                torch_dtype=torch.bfloat16,
                device_map="auto",
                trust_remote_code=True
            )

        if gradient_checkpointing:
            self.model.enable_input_require_grads()
            self.model.gradient_checkpointing_enable()

        self.freeze_bottom_layers(num_layers_to_freeze=8)
        self.learning_rate = learning_rate
        self.beta = beta
        self.total_steps = total_steps
        self.rouge_metric = ROUGEScore()
        self.bleu_metric = BLEUScore(n_gram=4, smooth=True)

        self.save_hyperparameters()
        self.validation_step_outputs = []

    def freeze_bottom_layers(self, num_layers_to_freeze):
        if hasattr(self.model, 'model') and hasattr(self.model.model, 'layers'):
            for i, layer in enumerate(self.model.model.layers):
                if i < num_layers_to_freeze:
                    for param in layer.parameters():
                        param.requires_grad = False
        print(f"Frozen {num_layers_to_freeze} bottom layers")

    def simpo_loss(self, chosen_logits, rejected_logits, chosen_labels, rejected_labels,
                   chosen_mask, rejected_mask, prompt_length):
        chosen_logprobs = self.get_logprobs(chosen_logits, chosen_labels, chosen_mask, prompt_length)
        rejected_logprobs = self.get_logprobs(rejected_logits, rejected_labels, rejected_mask, prompt_length)
        logit_diff = chosen_logprobs - rejected_logprobs
        loss = -F.logsigmoid(self.beta * logit_diff).mean()
        return loss, chosen_logprobs.mean(), rejected_logprobs.mean()

    def get_logprobs(self, logits, labels, attention_mask, prompt_length):
        shift_logits = logits[..., :-1, :].contiguous()
        shift_labels = labels[..., 1:].contiguous()
        shift_mask = attention_mask[..., 1:].contiguous()

        response_mask = torch.zeros_like(shift_mask)
        for i, plen in enumerate(prompt_length):
            if plen < shift_mask.size(1):
                response_mask[i, plen:] = shift_mask[i, plen:]

        # cast to float32 for numerical stability before log_softmax
        log_probs = F.log_softmax(shift_logits.float(), dim=-1)
        token_log_probs = torch.gather(log_probs, dim=-1, index=shift_labels.unsqueeze(-1)).squeeze(-1)

        masked_log_probs = token_log_probs * response_mask.float()
        valid_tokens = response_mask.sum(dim=-1)
        valid_tokens = torch.clamp(valid_tokens, min=1)

        return masked_log_probs.sum(dim=-1) / valid_tokens

    def forward(self, input_ids, attention_mask):
        return self.model(input_ids=input_ids, attention_mask=attention_mask)

    def training_step(self, batch, batch_idx):
        chosen_outputs = self(batch['chosen_input_ids'], batch['chosen_attention_mask'])
        rejected_outputs = self(batch['rejected_input_ids'], batch['rejected_attention_mask'])
        loss, chosen_logprobs, rejected_logprobs = self.simpo_loss(
            chosen_outputs.logits,
            rejected_outputs.logits,
            batch['chosen_input_ids'],
            batch['rejected_input_ids'],
            batch['chosen_attention_mask'],
            batch['rejected_attention_mask'],
            batch['prompt_length']
        )

        self.log('train_loss', loss, on_step=True, on_epoch=True, prog_bar=True, sync_dist=True)
        self.log('chosen_logprobs', chosen_logprobs, on_step=True, on_epoch=True, sync_dist=True)
        self.log('rejected_logprobs', rejected_logprobs, on_step=True, on_epoch=True, sync_dist=True)

        return loss

    def validation_step(self, batch, batch_idx):
        chosen_outputs = self(batch['chosen_input_ids'], batch['chosen_attention_mask'])
        rejected_outputs = self(batch['rejected_input_ids'], batch['rejected_attention_mask'])
        val_loss, chosen_logprobs, rejected_logprobs = self.simpo_loss(
            chosen_outputs.logits,
            rejected_outputs.logits,
            batch['chosen_input_ids'],
            batch['rejected_input_ids'],
            batch['chosen_attention_mask'],
            batch['rejected_attention_mask'],
            batch['prompt_length']
        )

        preds_text = []
        labels_text = []
        input_texts = []

        for i in range(len(batch['prompt'])):
            input_text = batch['input_text'][i]
            correct_output = batch['correct_output'][i]
            gen_prompt = f"Исправь все ошибки в предложении: {input_text}\nОтвет: "

            prompt_tokens = self.tokenizer(gen_prompt, return_tensors='pt', padding=False, truncation=True, max_length=150)
            prompt_tokens = {k: v.to(self.device) for k, v in prompt_tokens.items()}

            try:
                with torch.no_grad():
                    generated = self.model.generate(
                        **prompt_tokens,
                        max_new_tokens=32,
                        do_sample=False,
                        pad_token_id=self.tokenizer.pad_token_id,
                        eos_token_id=self.tokenizer.eos_token_id,
                        use_cache=True
                    )
                generated_text = self.tokenizer.decode(generated[0], skip_special_tokens=True)
                if gen_prompt in generated_text:
                    response = generated_text[len(gen_prompt):].strip()
                else:
                    response = generated_text.strip()
            except RuntimeError:
                response = input_text

            preds_text.append(response)
            labels_text.append(correct_output)
            input_texts.append(input_text)

        if batch_idx == 0:
            for i in range(min(len(preds_text), 3)):
                wandb.log({
                    f"val_example_{i}_input": input_texts[i],
                    f"val_example_{i}_predicted": preds_text[i],
                    f"val_example_{i}_reference": labels_text[i],
                })

        self.validation_step_outputs.append({
            'val_loss': val_loss,
            'preds_text': preds_text,
            'labels_text': labels_text,
            'chosen_logprobs': chosen_logprobs,
            'rejected_logprobs': rejected_logprobs
        })

        return {'val_loss': val_loss}

    def on_validation_epoch_end(self):
        all_preds = []
        all_labels = []
        all_losses = []
        all_chosen_logprobs = []
        all_rejected_logprobs = []

        for output in self.validation_step_outputs:
            all_preds.extend(output['preds_text'])
            all_labels.extend(output['labels_text'])
            all_losses.append(output['val_loss'])
            all_chosen_logprobs.append(output['chosen_logprobs'])
            all_rejected_logprobs.append(output['rejected_logprobs'])

        avg_loss = torch.stack(all_losses).mean() if all_losses else torch.tensor(0.0, device=self.device)
        avg_chosen_logprobs = torch.stack(all_chosen_logprobs).mean() if all_chosen_logprobs else torch.tensor(0.0, device=self.device)
        avg_rejected_logprobs = torch.stack(all_rejected_logprobs).mean() if all_rejected_logprobs else torch.tensor(0.0, device=self.device)
        rouge1 = rouge2 = rougeL = bleu_score = 0.0
        if all_preds and all_labels:
            valid_pairs = [(p, l) for p, l in zip(all_preds, all_labels) if p.strip() and l.strip()]
            if valid_pairs:
                valid_preds, valid_labels = zip(*valid_pairs)
                try:
                    rouge_score = self.rouge_metric(list(valid_preds), list(valid_labels))
                    rouge1 = rouge_score['rouge1_fmeasure'].mean().item()
                    rouge2 = rouge_score['rouge2_fmeasure'].mean().item()
                    rougeL = rouge_score['rougeL_fmeasure'].mean().item()
                except Exception as e:
                    print(f"ROUGE computation error: {e}")

                try:
                    bleu_score = self.bleu_metric(list(valid_preds), [[label] for label in valid_labels]).item()
                except Exception as e:
                    print(f"BLEU computation error: {e}")

        self.log('val_loss', avg_loss, on_epoch=True, prog_bar=True, sync_dist=True)
        self.log('val_chosen_logprobs', avg_chosen_logprobs, on_epoch=True, sync_dist=True)
        self.log('val_rejected_logprobs', avg_rejected_logprobs, on_epoch=True, sync_dist=True)
        self.log('rouge1', rouge1, on_epoch=True, prog_bar=True, sync_dist=True)
        self.log('rouge2', rouge2, on_epoch=True, prog_bar=True, sync_dist=True)
        self.log('rougeL', rougeL, on_epoch=True, prog_bar=True, sync_dist=True)
        self.log('bleu', bleu_score, on_epoch=True, prog_bar=True, sync_dist=True)
        self.validation_step_outputs.clear()

    def configure_optimizers(self):
        optimizer = AdamW(
            filter(lambda p: p.requires_grad, self.parameters()),
            lr=self.learning_rate,
            weight_decay=0.01
        )
        if self.total_steps:
            scheduler = get_cosine_schedule_with_warmup(
                optimizer,
                num_warmup_steps=int(0.1 * self.total_steps),
                num_training_steps=self.total_steps
            )
            return {
                "optimizer": optimizer,
                "lr_scheduler": {
                    "scheduler": scheduler,
                    "interval": "step",
                }
            }
        return optimizer


In [ ]:
def load_data(file_path_1, file_path_2):
    df = pd.read_csv(file_path_1)
    additional_df = pd.read_csv(file_path_2)

    df.rename(columns={'corrected': 'output', 'text': 'input'}, inplace=True)
    additional_df.rename(columns={'sent_correct': 'output', 'sent_corrupted': 'input'}, inplace=True)

    ctx_cols = [c for c in ['prev_sent', 'next_sent'] if c in df.columns]
    df = df[['input', 'output'] + ctx_cols].dropna(subset=['input', 'output'])
    for col in ['prev_sent', 'next_sent']:
        if col not in df.columns:
            df[col] = ''
    df = df.sample(frac=0.1, random_state=17)

    additional_df = additional_df[['input', 'output']].dropna()
    additional_df['prev_sent'] = ''
    additional_df['next_sent'] = ''
    additional_df = additional_df.sample(frac=0.05, random_state=17)

    combined_df = pd.concat([df, additional_df], ignore_index=True)
    combined_df['prev_sent'] = combined_df['prev_sent'].fillna('')
    combined_df['next_sent'] = combined_df['next_sent'].fillna('')
    combined_df = combined_df[combined_df['input'].str.strip() != '']
    combined_df = combined_df[combined_df['output'].str.strip() != '']
    return combined_df


In [ ]:
def calc_token_len(tokenizer, example):
    tokens = tokenizer(example, return_tensors="pt", truncation=True, max_length=192)["input_ids"][0]
    return len(tokens)


In [ ]:
tokenizer = AutoTokenizer.from_pretrained("Vikhrmodels/Vikhr-Gemma-2B-instruct")
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

In [ ]:
# Kaggle: attach 'sentences' dataset (sentences_full.csv from data_prep_new.ipynb
# for context-aware training, or sentences_preprocessed.csv for the baseline)
# and 'new-gen' dataset (synthetic_errors.csv)
SENTENCES_CSV = '/kaggle/input/sentences/sentences_full.csv'
SYNTHETIC_CSV = '/kaggle/input/new-gen/synthetic_errors.csv'

df = load_data(SENTENCES_CSV, SYNTHETIC_CSV)
df["input_token_len"] = df["input"].apply(lambda x: calc_token_len(tokenizer, x))
df = df[(df['input_token_len'] <= 148) & (df['input_token_len'] > 4)]
print(f"Dataset size after filtering: {len(df)}")


In [ ]:
train_df, val_df = train_test_split(df, test_size=0.1, shuffle=True, random_state=42)
print(f"Train size: {len(train_df)}, Validation size: {len(val_df)}")

train_dataset = PreferenceDataset(train_df, tokenizer, max_len=192)
val_dataset = PreferenceDataset(val_df, tokenizer, max_len=192)

In [ ]:
batch_size = 1 
num_epochs = 2 
accumulate_grad_batches = 8
steps_per_epoch = len(train_dataset) // batch_size
total_steps = steps_per_epoch * num_epochs
print(f"Total training steps: {total_steps}")

In [ ]:
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=0)

In [ ]:
model = SimPOLightningModule(
    model_name="Vikhrmodels/Vikhr-Gemma-2B-instruct",
    learning_rate=5e-5,
    beta=2.0,
    total_steps=total_steps,
    gradient_checkpointing=True,
    use_quantization=True
)

In [ ]:
trainer = Trainer(
    max_epochs=num_epochs,
    devices=1 if torch.cuda.is_available() else None,
    accelerator='gpu' if torch.cuda.is_available() else 'cpu',
    logger=pl.loggers.WandbLogger(project=WANDB_PROJECT),
    val_check_interval=1.0,
    gradient_clip_val=0.5,
    accumulate_grad_batches=accumulate_grad_batches,
    precision='bf16-true',
    enable_checkpointing=True,
    callbacks=[
        pl.callbacks.ModelCheckpoint(
            monitor='val_loss',
            mode='min',
            save_top_k=1,
            filename='simpo-gemma-{epoch:02d}-{val_loss:.3f}'
        ),
        pl.callbacks.EarlyStopping(
            monitor='val_loss',
            patience=2,
            mode='min',
            min_delta=0.01
        )
    ],
    enable_progress_bar=True,
    log_every_n_steps=50
)


In [ ]:
trainer.fit(model, train_loader, val_loader)

In [ ]:
def generate_predictions(model, tokenizer, val_df, output_csv_path, batch_size=1, max_samples=50):
    val_subset = val_df.head(max_samples)
    
    inputs_text = []
    preds_text = []
    references_text = []
    
    model.eval()
    device = next(model.parameters()).device
    for idx, row in tqdm(val_subset.iterrows(), total=len(val_subset), desc="Generating"):
        input_text = str(row['input'])
        correct_output = str(row['output'])
        prompt = f"Исправь все ошибки в предложении: {input_text}\nОтвет: "
        prompt_tokens = tokenizer(prompt, return_tensors='pt', padding=False, truncation=True, max_length=150)
        prompt_tokens = {k: v.to(device) for k, v in prompt_tokens.items()}
        
        try:
            with torch.no_grad():
                generated = model.model.generate(
                    **prompt_tokens,
                    max_new_tokens=32, 
                    do_sample=False,
                    pad_token_id=tokenizer.pad_token_id,
                    eos_token_id=tokenizer.eos_token_id,
                    use_cache=True
                )
        except RuntimeError as e:
            print(f"Generation error for sample {idx}: {e}")
            response = input_text
        else:
            generated_text = tokenizer.decode(generated[0], skip_special_tokens=True)
            if prompt in generated_text:
                response = generated_text[len(prompt):].strip()
            else:
                response = generated_text.strip()
        
        inputs_text.append(input_text)
        preds_text.append(response)
        references_text.append(correct_output)
        if idx % 10 == 0:
            torch.cuda.empty_cache()
    
    results_df = pd.DataFrame({
        "input_text": inputs_text,
        "predicted_text": preds_text,
        "reference_text": references_text
    })
    results_df.to_csv(output_csv_path, index=False, encoding='utf-8')
    print(f"Predictions saved to {output_csv_path}")
    print(f"Generated {len(results_df)} predictions")
    for i in range(min(5, len(results_df))):
        print(f"\nExample {i+1}:")
        print(f"Input: {results_df.iloc[i]['input_text']}")
        print(f"Reference: {results_df.iloc[i]['reference_text']}")
        print(f"Predicted: {results_df.iloc[i]['predicted_text']}")
    
    return results_df

In [ ]:
results_df = generate_predictions(model, tokenizer, val_df, 'simpo_output.csv')
model.model.save_pretrained('simpo_gemma_grammar_correction')
tokenizer.save_pretrained('simpo_gemma_grammar_correction')

try:
    rouge_metric = ROUGEScore()
    bleu_metric = BLEUScore(n_gram=4, smooth=True)
    
    valid_results = results_df[
        (results_df['predicted_text'].str.strip() != '') & 
        (results_df['reference_text'].str.strip() != '')
    ]
    
    if not valid_results.empty:
        rouge_scores = rouge_metric(
            valid_results['predicted_text'].tolist(), 
            valid_results['reference_text'].tolist()
        )
        bleu_score = bleu_metric(
            valid_results['predicted_text'].tolist(), 
            [[ref] for ref in valid_results['reference_text'].tolist()]
        )
        
        print(f"ROUGE-1: {rouge_scores['rouge1_fmeasure'].mean():.4f}")
        print(f"ROUGE-2: {rouge_scores['rouge2_fmeasure'].mean():.4f}")
        print(f"ROUGE-L: {rouge_scores['rougeL_fmeasure'].mean():.4f}")
        print(f"BLEU: {bleu_score:.4f}")
    else:
        print("No valid predictions")
except Exception as e:
    print(f"Error computing final metrics: {e}")

In [ ]:
del model
torch.cuda.empty_cache()
gc.collect()
wandb.finish()